# Fix hidden-slide page numbers in Unstructured output

When a PPTX goes through the **VLM partitioner**, hidden slides are dropped and the remaining slides are renumbered 1..N, so page numbers no longer match the deck.

This notebook:
1. Reads the deck and finds hidden slides
2. Builds a *processed page → real slide* map
3. Rewrites page numbers in the Partitioner and Chunker JSONs (`page_number`, `data-page-number` in `text_as_html`, and the same inside `orig_elements`)
4. Verifies the result and checks that no chunk spans more than one slide

Original files are not modified; corrected copies are saved as `*-fixed.json`.

In [ ]:
%pip install -q python-pptx

## 1. Set your file paths

In [ ]:
from pathlib import Path

FOLDER = Path(r"C:\Users\sdhar2\Downloads\ZURZUVAE Creative CTV Testing - Reports Study Data\ER PPD JSONs")

PPTX_FILE        = FOLDER / "ZUR-Creative-Optimization_Qualitative-Topline-Report_5125F-48c3b65e.pptx"
PARTITIONER_JSON = FOLDER / "ZUR-Creative-Optimization_Qualitative-Topline-Report_5125F-48c3b65e-Partitioner-JSON.json"
CHUNKER_JSON     = FOLDER / "ZUR-Creative-Optimization_Qualitative-Topline-Report_5125F-48c3b65e-Chunker-JSON.json"

for p in (PPTX_FILE, PARTITIONER_JSON, CHUNKER_JSON):
    print(("OK      " if p.exists() else "MISSING ") + p.name)

## 2. Helper functions

In [ ]:
import base64, gzip, json, re
from pptx import Presentation

PAGE_RE = re.compile(r'(data-page-number=\\?")(\d+)(\\?")')

def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def save(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def read_orig(md):
    """Return (list_of_orig_elements, was_compressed)."""
    orig = md.get("orig_elements") or []
    if isinstance(orig, str):   # base64-encoded gzipped JSON
        return json.loads(gzip.decompress(base64.b64decode(orig))), True
    return orig, False

def slide_map(pptx_path):
    prs = Presentation(pptx_path)
    hidden, mapping, visible_idx = [], {}, 0
    for n, slide in enumerate(prs.slides, start=1):
        if slide._element.get("show") in ("0", "false"):
            hidden.append(n)
        else:
            visible_idx += 1
            mapping[visible_idx] = n
    return len(prs.slides), hidden, mapping

def fix_element(el, mapping, missing):
    md = el.get("metadata") or {}
    pn = md.get("page_number")
    if pn is not None:
        if pn in mapping:
            md["page_number"] = mapping[pn]
        else:
            missing.add(pn)
    html = md.get("text_as_html")
    if isinstance(html, str):
        md["text_as_html"] = PAGE_RE.sub(
            lambda m: f"{m.group(1)}{mapping.get(int(m.group(2)), int(m.group(2)))}{m.group(3)}", html)
    if md.get("orig_elements"):
        items, compressed = read_orig(md)
        for sub in items:
            fix_element(sub, mapping, missing)
        md["orig_elements"] = (base64.b64encode(gzip.compress(json.dumps(items).encode())).decode()
                               if compressed else items)

def pages_in_chunk(ch):
    md = ch.get("metadata") or {}
    pages = {int(p) for p in PAGE_RE.findall(md.get("text_as_html") or "") for p in [p[1]]}
    items, _ = read_orig(md)
    pages |= {(e.get("metadata") or {}).get("page_number") for e in items} - {None}
    return sorted(pages)

## 3. Find hidden slides

Check the hidden list. With slide 20 labelled as page 17, expect **three** hidden slides before slide 20. If only 8 and 10 show up, stop: the shift has another cause.

In [ ]:
total, hidden, mapping = slide_map(PPTX_FILE)
print(f"Deck: {total} slides | hidden: {hidden or 'none'} | visible: {len(mapping)}")
print("\nProcessed page -> real slide (only where different):")
for p, s in mapping.items():
    if p != s:
        print(f"  page {p:>3} -> slide {s}")

## 4. Fix both JSONs

A file is skipped if its highest page number is above the number of visible slides, because then hidden slides were *not* dropped and remapping would make it wrong.

In [ ]:
fixed_paths = {}
for label, path in [("Partitioner", PARTITIONER_JSON), ("Chunker", CHUNKER_JSON)]:
    elements = load(path)
    pages = {(e.get("metadata") or {}).get("page_number") for e in elements} - {None}
    top = max(pages) if pages else 0
    if not hidden:
        print(f"{label}: no hidden slides, nothing to fix"); continue
    if top > len(mapping):
        print(f"{label}: SKIPPED - highest page {top} > {len(mapping)} visible slides. Check manually.")
        continue
    missing = set()
    for el in elements:
        fix_element(el, mapping, missing)
    out = path.with_name(path.stem + "-fixed.json")
    save(elements, out)
    fixed_paths[label] = out
    print(f"{label}: {len(elements)} items -> saved {out.name}")
    if missing:
        print(f"  WARNING: page numbers with no matching slide: {sorted(missing)}")

## 5. Verify

Page numbers in the fixed Partitioner JSON should never be a hidden slide, and every chunk should come from exactly one slide (true only if the Chunker uses **Chunk by page**).

In [ ]:
if "Partitioner" in fixed_paths:
    parts = load(fixed_paths["Partitioner"])
    used = sorted({(e.get("metadata") or {}).get("page_number") for e in parts} - {None})
    clash = set(used) & set(hidden)
    print(f"Partitioner pages now: {used[0]}..{used[-1]} ({len(used)} pages)")
    print("Hidden slides used as page numbers:", sorted(clash) if clash else "none - good")

if "Chunker" in fixed_paths:
    chunks = load(fixed_paths["Chunker"])
    bad = [(i, ch["metadata"].get("page_number"), pages_in_chunk(ch), (ch.get("text") or "")[:70])
           for i, ch in enumerate(chunks, 1) if len(pages_in_chunk(ch)) > 1]
    has_orig = any((ch.get("metadata") or {}).get("orig_elements") for ch in chunks)
    print(f"\nChunks: {len(chunks)} | orig_elements present: {has_orig} | spanning >1 slide: {len(bad)}")
    for i, labelled, pages, text in bad:
        print(f"  chunk #{i}: labelled {labelled}, contains slides {pages} | {text!r}")

## 6. Spot-check a slide

Change `SLIDE` to any slide number and compare the text with the deck.

In [ ]:
SLIDE = 20
if "Chunker" in fixed_paths:
    for ch in load(fixed_paths["Chunker"]):
        if ch["metadata"].get("page_number") == SLIDE:
            print(f"--- slide {SLIDE} ---")
            print((ch.get("text") or "")[:500], "\n")